# Answer-only SFT and optional DPO


Select **Runtime → Change runtime type → A100 GPU**. Run cells in order.
All artifacts live under `/content/drive/MyDrive/YLM_runs/<RUN_NAME>`.
Fixed names intentionally resume; change the name for a new experiment. Completed
`metrics.json` arms are skipped. Do not change budgets/model/seeds while resuming.

`src/profiling.py` records throughput, phase totals and per-step times, peak CUDA
memory, GPU utilization, a bounded `torch.profiler` trace, and `summary.json`.
GPU telemetry can be unavailable; profiles cover the script's profiled region,
not setup/download time. Reruns overwrite the individual invocation's profile.

**Time estimates (not measured on Colab):** setup and label reuse may take 5–20 minutes; allow roughly 30 minutes–4 hours for all six SFT arms at 100k tokens. Optional DPO may add 1–6 hours and higher memory use because it loads a reference model. These are broad planning estimates. DPO counts policy and reference tokens, so equal SFT/DPO budgets do not imply equal examples. Default QM9-only SFT avoids task-specific downstream exposure.

In [ ]:
RUN_NAME = 'ylm_lm_posttrain' #@param {type:"string"}
PRETRAIN_RUN_NAME = 'ylm_lm_pretrain_ablation' #@param {type:"string"}
SEEDS = [0]
ARMS = ['smiles', 'smiles_xyz', 'prefixA', 'xattnA', 'xattnB', 'xattnC']
SOURCES = ['qm9']  # optionally add 'calisol', 'reddb'; preserve downstream split seeds
SFT_TOKENS_BUDGET = 100_000
ENABLE_DPO = False #@param {type:"boolean"}
DPO_TOKENS_BUDGET = 100_000
BATCH_SIZE = 2
MAX_LEN = 512
CKPT_EVERY = 50
INSTALL_MP_API = False #@param {type:"boolean"}
SETTINGS = dict(pretrain_run=PRETRAIN_RUN_NAME, seeds=SEEDS, arms=ARMS, sources=SOURCES,
                sft_tokens=SFT_TOKENS_BUDGET, dpo=ENABLE_DPO, dpo_tokens=DPO_TOKENS_BUDGET,
                batch_size=BATCH_SIZE, max_len=MAX_LEN, lr=1e-5, beta=0.1)

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, shutil, urllib.request, zipfile

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/YLM_runs')
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME
SAVE = BASE / RUN_NAME
SAVE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/YLM')
BRANCH = 'feat/llm-irreps-ablation-oct30'
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/yaniguan/YLM.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
packages = ['e3nn==0.6.0', 'numpy<2.3', 'scipy', 'rdkit', 'tblite',
            'transformers>=4.40,<5', 'accelerate', 'scikit-learn', 'pynvml', 'pytest']
if INSTALL_MP_API:
    packages.append('mp-api')
subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)
import torch
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print('GPU:', torch.cuda.get_device_name(), '| output:', SAVE)
if 'A100' not in torch.cuda.get_device_name():
    print('Warning: estimates assume A100; this runtime has a different GPU.')

def run_script(script, *args):
    command = [sys.executable, str(ROOT / 'scripts' / script), *map(str, args)]
    print(' '.join(command), flush=True)
    with (SAVE / 'commands.log').open('a') as log:
        log.write(' '.join(command) + '\n')
    subprocess.run(command, cwd=ROOT, check=True)

# Preserve experiment settings on reruns; use a new RUN_NAME for changed settings.
settings = dict(SETTINGS, branch=BRANCH)
manifest = SAVE / 'notebook_config.json'
if manifest.exists():
    assert json.loads(manifest.read_text()) == settings, 'Settings changed: choose a new RUN_NAME.'
else:
    manifest.write_text(json.dumps(settings, indent=2))
(SAVE / 'git_commit.txt').write_text(subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True))

In [ ]:
PRETRAIN = BASE / PRETRAIN_RUN_NAME
assert PRETRAIN != SAVE
assert PRETRAIN.exists(), 'Run the pretraining notebook first.'
LABEL_RUN = PRETRAIN

# Keep the absolute local data paths stable across Colab sessions: saved configs
# use /content/YLM/data/... and posttraining reads those same paths.
data = ROOT / 'data'
data.mkdir(exist_ok=True)
if not (data / 'gdb9.sdf').exists():
    archive = data / 'qm9.zip'
    if not archive.exists():
        temporary = archive.with_suffix('.download')
        urllib.request.urlretrieve(
            'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/molnet_publish/qm9.zip', temporary)
        temporary.replace(archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(data)
assert (data / 'gdb9.sdf').exists()
label_cache = LABEL_RUN / 'qm9_xtb.pkl'
if label_cache.exists():
    shutil.copy2(label_cache, data / 'qm9_xtb.pkl')
else:
    run_script('01_make_labels.py', '--n', 20000, '--workers', os.cpu_count() or 2,
               '--seed', 0, '--sdf', data / 'gdb9.sdf', '--out', data / 'qm9_xtb.pkl')
    label_cache.parent.mkdir(parents=True, exist_ok=True)
    temporary = label_cache.with_suffix('.tmp')
    shutil.copy2(data / 'qm9_xtb.pkl', temporary)
    temporary.replace(label_cache)
if label_cache != SAVE / 'qm9_xtb.pkl' and not (SAVE / 'qm9_xtb.pkl').exists():
    shutil.copy2(label_cache, SAVE / 'qm9_xtb.pkl')

All SFT/DPO calls pass `--resume`, which starts fresh when no saved state exists. Periodic and final checkpoints stay in Drive. DPO starts from SFT and never modifies its source checkpoint.

In [ ]:
# Persist public source files, provenance and the optional MP cache on Drive.
public = SAVE / 'data' / 'electrolyte'
public.mkdir(parents=True, exist_ok=True)
local = ROOT / 'data' / 'electrolyte'
local.parent.mkdir(exist_ok=True)
if local.is_symlink():
    if local.resolve() != public.resolve():
        local.unlink()
        local.symlink_to(public, target_is_directory=True)
elif local.exists():
    shutil.copytree(local, public, dirs_exist_ok=True)
    shutil.rmtree(local)
    local.symlink_to(public, target_is_directory=True)
else:
    local.symlink_to(public, target_is_directory=True)
run_script('download_public_data.py', '--out', public)

In [ ]:
for seed in SEEDS:
    for arm in ARMS:
        name = f'{arm}_seed{seed}'
        source = PRETRAIN / 'pretrain' / name
        assert (source / 'metrics.json').exists(), f'Finish pretraining: {source}'
        for stage in (['sft', 'dpo'] if ENABLE_DPO else ['sft']):
            out = SAVE / stage / name
            if not (out / 'metrics.json').exists():
                budget = SFT_TOKENS_BUDGET if stage == 'sft' else DPO_TOKENS_BUDGET
                run_script('21_posttrain.py', '--pretrained', source, '--out', out,
                           '--sources', *SOURCES, '--tokens_budget', budget, '--seed', seed,
                           '--batch_size', BATCH_SIZE, '--max_len', MAX_LEN, '--lr', 1e-5,
                           '--beta', 0.1, '--ckpt_every', CKPT_EVERY, '--device', 'cuda', '--resume',
                           '--profile_dir', SAVE / 'profiling' / f'{stage}_{name}',
                           *(['--dpo'] if stage == 'dpo' else []))
            else:
                print('Skip completed:', stage, name)
            source = out  # DPO starts from the completed SFT checkpoint

In [ ]:
# Final cell: retain individual summaries and aggregate without including itself.
profile_root = SAVE / 'profiling'
profile_root.mkdir(exist_ok=True)
summaries = {}
for path in sorted(profile_root.rglob('summary.json')):
    if path != profile_root / 'summary.json':
        summaries[str(path.parent.relative_to(profile_root))] = json.loads(path.read_text())
combined = {'runs': summaries, 'run_count': len(summaries),
            'note': 'Individual summaries describe the latest invocation, not cumulative resumed sessions.'}
temporary = profile_root / 'summary.tmp'
temporary.write_text(json.dumps(combined, indent=2))
temporary.replace(profile_root / 'summary.json')
print('Profiling:', profile_root / 'summary.json')